## Ch9-02 -- Evidence sufficiency, applied to two real records

This notebook introduces Hawkins' sufficiency check (`uv run python -m glossary tutorial sufficiency`), applied to two real `ReviewRecord`s already built earlier in this tutorial; after running it you can tell, for each, whether its own `counterevidence` and `residual_uncertainties` are genuinely substantive and whether its `engineering_conclusion` honestly matches what its own evidence supports.

This tutorial has no central registry of every `ReviewRecord` it has ever built: each chapter's notebook constructs its own records as local Python objects, per the construction-zone pattern (`toaster-review-protocol`), with nothing shared beyond the file each one lives in. So this notebook does not scan "every record the tutorial has ever produced" -- that would need a real registry, which does not exist and is out of this chapter's scope (see this chapter's own report for the search that confirmed it). Instead it reconstructs two real records faithfully, field for field, from `chapters/ch06-recursive-decomp/02-second-level.ipynb` (`AS-C06`, a mechanism-selection judgment argued from a domain premise) and `chapters/ch08-checking/02-violation-witness.ipynb` (`AS-C08`, grounded in Chapter 8's real Z3 proof), and demonstrates what a sufficiency check on each one actually looks like. What follows demonstrates the mechanics of the check on a small, representative sample, not an exhaustive audit of every judgment record this tutorial has ever produced.

In [1]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch08-cumulative.sysml").read_text()
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"


Before reconstructing two real records, the mechanized half of a sufficiency check: `validate_record()` already refuses a record whose `counterevidence` field is empty outright, distinct from asking whether a populated field is substantive, which is this notebook's own further question.

In [2]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

blank_counterevidence = ReviewRecord(
    identifier="AS-BAD",
    kind="asserted_solution",
    claim="Some claim",
    model_ref="ToasterDemo",
    content_hash=hash_content(source),
    scope="Some scope",
    criteria="Some criteria",
    rationale="Some rationale",
    counterevidence="",   # intentionally empty
    record_kind="worked_example",
)
errors = validate_record(blank_counterevidence)
assert len(errors) > 0, "Expected validation to fail on empty counterevidence"
print(f"Negative control ok: errors={errors}")


Negative control ok: errors=['counterevidence is empty']


`AS-C06`, rebuilt below exactly as Chapter 6 notebook 02 built it: a mechanism-selection judgment, argued from a domain premise about how a resistive element and a combustion burner each respond to a discrete timing signal, not from anything the model itself yet connects. Its `content_hash` is computed against `models/ch06-cumulative.sysml`, the real model it was actually written against, not against the current model this chapter uses -- the same file it was built from, unchanged, so the reconstruction is faithful to what that chapter actually recorded.

In [3]:
ch06_source = Path("../../models/ch06-cumulative.sysml").read_text()

as_c06 = ReviewRecord(
    identifier="AS-C06",
    kind="asserted_solution",
    claim=(
        "ResistanceCoil, an electrically switched resistive element that converts "
        "energy to heat by Joule heating, is selected over a combustion-based "
        "alternative (a gas burner, the tongs-and-blowtorch alternative this "
        "tutorial already contrasts) as the mechanism HeatGenerator commits to."
    ),
    model_ref="ToasterDemo::ResistanceCoil",
    content_hash=hash_content(ch06_source),
    scope="ToasterDemo::HeatGenerator and its realizations",
    criteria=(
        "The chosen mechanism must pair with the discrete timing control "
        "ControlSystem's durationOut already provides, and must expose a rating "
        "heatGenerationReq's power threshold can be checked against once a "
        "concrete part exists."
    ),
    premises=[
        "ControlSystem declares durationOut : DurationPort (Chapter 5), a "
        "discrete duration signal, confirmed by model.find() in Chapter 6.",
        "Domain premise, not derived from the model: a resistive element "
        "responds to being switched on and off directly, while a combustion "
        "source needs separate ignition and fuel-metering machinery to do "
        "the same. Neither HeatGenerator nor any of its realizations is yet "
        "connected to ControlSystem's port in this model; this premise is "
        "about the physical mechanisms themselves, not about what the model "
        "currently wires together.",
    ],
    assumption_refs=[
        "HeatGenerator::energyIn and GenerateHeat carry no energy-form commitment "
        "(Chapter 6 notebook 01): this record is what actually commits to an "
        "electrical form, not a fact already built into the port or the function."
    ],
    evidence_refs=[
        "model.find('ToasterDemo::ControlSystem::durationOut') resolves to a "
        "real PortUsage, confirmed in Chapter 6 notebook 02.",
    ],
    rationale=(
        "An electrically resistive element responds to being switched on "
        "and off directly, the same shape as duration's discrete timing "
        "signal, while a combustion-based burner needs separate ignition "
        "and fuel-metering machinery to respond the same way (the domain "
        "premise above). That is a claim about how the two mechanisms "
        "work, not something this model currently shows: no realization "
        "of HeatGenerator is yet connected to ControlSystem's "
        "durationOut port, so this selection is a reasoned engineering "
        "preference argued from mechanism, not a claim that the model "
        "already connects one mechanism and not the other."
    ),
    counterevidence=(
        "This does not rule out a combustion design: a burner controlled by its "
        "own timed valve could equally use a duration-like signal, which is "
        "exactly why the argument above rests on a domain premise about how "
        "the two mechanisms work, not on anything the model itself already "
        "builds or connects. Joule heating's own relation (power proportional "
        "to resistance and the square of current) is still not modeled, so "
        "efficiency and response-time comparisons remain out of reach "
        "either way."
    ),
    residual_uncertainties=(
        "Once a supply and a control policy are modeled together, this "
        "selection could be revisited against a real trade study rather than "
        "a domain premise alone."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06)
print(f"AS-C06 validation errors: {errors}")


AS-C06 validation errors: []


`AS-C08`, rebuilt below exactly as Chapter 8 notebook 02 built it: the record grounded in the chapter's real Z3 proof of `deliveredEnergyBoundedBySupply`. Its `content_hash` is computed against `models/ch08-cumulative.sysml` -- the same file this chapter's own notebooks already load, since Chapter 8 is the real, current model.

In [4]:
as_c08 = ReviewRecord(
    identifier="AS-C08",
    kind="asserted_solution",
    claim=(
        "deliveredEnergyBoundedBySupply, a hand-restated real-arithmetic lemma of the same "
        "shape as HeatGenerator's efficiencyBounded constraint and deliveredEnergy's own "
        "definition, holds for every value of efficiency in [0,1] and every non-negative "
        "power and duration a companion restatement admits."
    ),
    model_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    content_hash=hash_content(source),
    scope=(
        "The lemma governs the companion restatement's own heatGenCheck.efficiency, "
        "heatGenCheck.power and heatGenCheckDuration features; it is not a solver-checked "
        "reference to HeatGenerator's own efficiencyBounded or deliveredEnergy (DEFERRED.md "
        "D-030, D-031), only a hand-restated copy of the same shape."
    ),
    criteria=(
        "verify_holds() reports a single satisfied verdict for deliveredEnergyBoundedBySupply, "
        "with the reason text naming z3 (not propagation alone), proved for all values of the "
        "unbound heatGenCheck.efficiency, heatGenCheck.power and heatGenCheckDuration features."
    ),
    evidence_refs=[
        "verify_holds: deliveredEnergyBoundedBySupply satisfied (z3, Chapter 8 notebook 02)."
    ],
    rationale=(
        "verify_holds() calls sysml-toolkit's real verify --solve command, which runs Z3 "
        "over the unbound features of a companion restatement of this lemma and reports "
        "deliveredEnergyBoundedBySupply as satisfied: proved for all values the restatement "
        "admits, not read back from one entered value. This is a materially different kind of "
        "evidence from an evaluate-only verdict: verify_satisfaction() could only ever check "
        "a relation at whichever single power, duration and efficiency a candidate happens to "
        "carry."
    ),
    counterevidence=(
        "This proof is NOT a solver-checked reference to HeatGenerator's own "
        "efficiencyBounded constraint or deliveredEnergy calc: this toolchain's Z3 backend "
        "does not compose two separately declared assert constraints, whether sibling or "
        "inherited (DEFERRED.md D-030), and cannot reason through a chained calc invocation "
        "like heatGenCheck.deliveredEnergy(...) (D-031). Confirmed directly in Chapter 8: "
        "loosening efficiencyBounded's own literal bound to <= 1.5, or doubling "
        "deliveredEnergy's own definition by a factor of 2.0, in the committed model changes "
        "neither the original elements' own verdicts nor this lemma's verdict at all, because "
        "the lemma restates its own copy of both rather than referencing either."
    ),
    residual_uncertainties=(
        "Whether efficiency, power and duration ever take values outside the bound in a "
        "real candidate is not addressed by this proof; it establishes only that the "
        "restated lemma respects conservation wherever its own bound is honored. If "
        "HeatGenerator's own efficiencyBounded or deliveredEnergy is ever edited, this "
        "record's content_hash (computed from the whole model file) does go stale, which "
        "forces a re-review, but nothing automatically re-checks that the restated copy "
        "still matches the edited original; that check would be manual. No physical heat "
        "generator has been checked against this property."
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08)
print(f"AS-C08 validation errors: {errors}")


AS-C08 validation errors: []


Both records validate cleanly. The sufficiency question goes further than `validate_record()` does: Hawkins' idea (`uv run python -m glossary tutorial sufficiency`) asks whether the premises named are enough to establish the conclusion's probable truth, which means reading `counterevidence` and `residual_uncertainties` for whether they are substantive -- naming a specific, checkable limit -- rather than placeholder text that would validate regardless of content.

In [5]:
def word_count(text: str) -> int:
    return len(text.split())

for record in (as_c06, as_c08):
    print(f"{record.identifier}:")
    print(f"  counterevidence ({word_count(record.counterevidence)} words): {record.counterevidence[:80]}...")
    print(f"  residual_uncertainties ({word_count(record.residual_uncertainties)} words): {record.residual_uncertainties[:80]}...")
    print(f"  disposition: {record.disposition}")
    print(f"  engineering_conclusion: {record.engineering_conclusion}")


AS-C06:
  counterevidence (78 words): This does not rule out a combustion design: a burner controlled by its own timed...
  residual_uncertainties (26 words): Once a supply and a control policy are modeled together, this selection could be...
  disposition: pending
  engineering_conclusion: undetermined
AS-C08:
  counterevidence (97 words): This proof is NOT a solver-checked reference to HeatGenerator's own efficiencyBo...
  residual_uncertainties (89 words): Whether efficiency, power and duration ever take values outside the bound in a r...
  disposition: pending
  engineering_conclusion: supported


Both records read as substantive, not placeholder: `AS-C06`'s counterevidence names a specific competing design (a valve-controlled burner) and a specific unmodeled relation (Joule heating), not a generic hedge; `AS-C08`'s counterevidence names two specific, cited toolchain limits (`DEFERRED.md` D-030, D-031) and the exact edits that were tried and failed to move the lemma's verdict. Where the two records differ most is what `engineering_conclusion` claims. `AS-C06` stays `undetermined` even though the record does make a selection: its own counterevidence admits the selection is argued from a domain premise, not a trade study, so `undetermined` is the honest reading of what the evidence actually supports, not underclaiming. `AS-C08` is `supported`: because its own `claim` field is already narrowed to the hand-restated lemma, not to `HeatGenerator`'s original elements, the thing that actually got a Z3 proof is exactly what the record claims was established. Sufficiency here is a judgment about whether the named premises would make the conclusion's probable truth follow (Hawkins et al. 2011, SS3.1), and both records name real, specific reasons their own conclusion still might not, which is what makes each one checkable rather than merely asserted.

In [6]:
assert as_c06.disposition != "accepted" and as_c08.disposition != "accepted"
assert as_c06.counterevidence.strip() and as_c08.counterevidence.strip()
assert as_c06.residual_uncertainties.strip() and as_c08.residual_uncertainties.strip()
assert as_c06.engineering_conclusion == "undetermined"
assert as_c08.engineering_conclusion == "supported"

print("Both records: disposition != \"accepted\" (SA-7); counterevidence and residual_uncertainties non-empty and substantive.")
print(f"AS-C06 engineering_conclusion={as_c06.engineering_conclusion!r} (selection argued from a domain premise, not a trade study)")
print(f"AS-C08 engineering_conclusion={as_c08.engineering_conclusion!r} (claim already narrowed to the hand-restated lemma that was actually proved)")
conn.close()


Both records: disposition != "accepted" (SA-7); counterevidence and residual_uncertainties non-empty and substantive.
AS-C06 engineering_conclusion='undetermined' (selection argued from a domain premise, not a trade study)
AS-C08 engineering_conclusion='supported' (claim already narrowed to the hand-restated lemma that was actually proved)


The two records rebuilt above, field for field from their own chapters, both validate cleanly, and reading their actual counterevidence and residual_uncertainties (not just checking they are non-empty) shows two different, both honest, levels of confidence -- exactly what a reader would need to decide how much to lean on each one.

Try the chapter exercise in `exercises/ch09/exercise.ipynb`: it asks you to produce a coverage table for the bread-handling requirements, using the query-and-join pattern notebook 01 builds.